# 🤖 MiniMind-V VLA 視覺語言動作智能體：Google Colab 全流程訓練（支援 MoE 混合專家）
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Child-pi/minimind-v-vla/blob/VLA/minimind_v_vla_moe_colab.ipynb)

> **GitHub Repository**: [Child-pi/minimind-v-vla](https://github.com/Child-pi/minimind-v-vla)
> 
> 本專案實作了具備端到端決策能力的 **VLA (Vision-Language-Action)** 井字棋智能體：
> - 📸 **視覺感知 (Vision)**：接收 3x3 棋盤渲染圖像並提取語意表徵
> - 💭 **思維鏈推理 (CoT Reasoning)**：輸出局勢評估與戰術思考文字
> - 🎯 **動作決策 (Action)**：直接輸出離散落子位置 (0~8)
> 
> 🧠 **全架構雙模支援（MoE 混合專家 vs. Dense 密集模型）**：
> - **MoE 混合專家模式**：採用 **1 個共享專家 (Shared Expert) + 4 個路由專家 (Routed Experts, Top-2 激活)**，每次推理僅動用部分算力，兼顧高容量與極速推理！
> - **Dense 密集模型模式**：標準 768 維單一前饋網路，適合直接加載官方預訓練權重。
> - **斷點續訓 (Resume Training) & 自我修復**：支援斷線打撈、SFT 續訓、PPO 增量訓練與 Google Drive 定時同步。


## ⚙️ 步驟 1：環境設定、Google Drive 掛載與雲端權重自動打撈
- 檢查 GPU 資源 (建議使用 T4、V100 或 A100)
- 掛載 Google Drive，設定權重持久化目錄
- **自我修復機制 (Self-Healing)**：若先前的訓練曾在 Google Drive 留下 Checkpoint，自動偵測並同步至執行環境，無縫接軌繼續訓練


In [ ]:
# 檢查 GPU 環境
!nvidia-smi

import os
import sys
import time
import shutil
import glob

# 設定環境變數：防止 CUDA 顯存碎片 OOM、停用 Tokenizer 多進程告警、Pygame 無頭模式
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["WANDB_MODE"] = "offline"  # 避免無 wandb 帳號時卡在終端輸入

# 掛載 Google Drive (建議開啟以永久保存 Checkpoint 並支援斷點續訓)
MOUNT_DRIVE = True # @param {type:"boolean"}
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/minimind_v_vla_checkpoints"

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
        print(f"✅ Google Drive 掛載成功，備份路徑: {DRIVE_PROJECT_DIR}")
    except Exception as e:
        print(f"⚠️ Google Drive 掛載失敗或跳過: {e}")


## 📦 步驟 2：克隆專案倉庫、安裝相依套件與 MoE 補丁檢查
克隆倉庫並安裝套件，同時自動確認訓練腳本具備 `--use_moe` 混合專家與 `--resume_from` 斷點續訓能力。


In [ ]:
# 克隆程式碼倉庫
REPO_URL = "https://github.com/Child-pi/minimind-v-vla.git"
PROJECT_DIR = "/content/minimind-v-vla"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    print("✅ 倉庫克隆完成！")
else:
    print("ℹ️ 倉庫已存在，跳過克隆。")

%cd {PROJECT_DIR}

# 安裝依賴套件
!pip install -q -r requirements.txt
!pip install -q pygame gymnasium pillow numpy wandb tqdm

# 建立輸出目錄
os.makedirs("VLA/data", exist_ok=True)
os.makedirs("VLA/models", exist_ok=True)
os.makedirs("VLA/result", exist_ok=True)
os.makedirs("model/vision_model", exist_ok=True)

# 自我修復：自動從 Google Drive 還原上次訓練的權重
if os.path.exists(DRIVE_PROJECT_DIR):
    drive_ckpts = glob.glob(f"{DRIVE_PROJECT_DIR}/*.pth")
    if drive_ckpts:
        print(f"🔍 在 Google Drive 發現 {len(drive_ckpts)} 個歷史權重，自動還原至本機以支援繼續訓練：")
        for ckpt in drive_ckpts:
            dst = os.path.join("VLA/models", os.path.basename(ckpt))
            if not os.path.exists(dst):
                shutil.copy2(ckpt, dst)
                print(f"  📥 還原: {os.path.basename(ckpt)} -> {dst}")
            else:
                print(f"  ℹ️ 本機已存在: {os.path.basename(ckpt)}")


## 📥 步驟 3：模型架構選擇 (MoE vs. Dense) 與基座權重下載
在此步驟中您可以選擇訓練 **MoE (混合專家模型)** 或 **Dense (傳統密集模型)**：
- **MoE 架構**：啟用混合專家層（預設 1 個共享專家 + 4 個路由專家，Top-2 激活）。若選擇 MoE，系統會自動下載 MiniMind2-MoE 基座權重以利快速收斂。
- **Dense 架構**：標準 768 維單一前饋模型，使用 MiniMind2-V 基座。


In [ ]:
import os

# 選擇模型架構
MODEL_ARCH = "MoE" # @param ["MoE", "Dense"]
NUM_EXPERTS_PER_TOK = 2 # @param {type:"integer"}
N_ROUTED_EXPERTS = 4 # @param {type:"integer"}
N_SHARED_EXPERTS = 1 # @param {type:"integer"}

print(f"🎯 目標模型架構: {MODEL_ARCH}")
if MODEL_ARCH == "MoE":
    print(f"   - 路由專家總數: {N_ROUTED_EXPERTS}")
    print(f"   - 每個 Token 激活專家數 (Top-K): {NUM_EXPERTS_PER_TOK}")
    print(f"   - 共享專家數: {N_SHARED_EXPERTS}")

# 1. 下載 CLIP 視覺編碼器
CLIP_DIR = "/content/minimind-v-vla/model/vision_model/clip-vit-base-patch16"
if not os.path.exists(CLIP_DIR) or len(os.listdir(CLIP_DIR)) == 0:
    print("📥 正在下載 CLIP 視覺模型 (clip-vit-base-patch16)...")
    !git clone --depth=1 https://huggingface.co/openai/clip-vit-base-patch16 {CLIP_DIR}
    print("✅ CLIP 下載完成！")
else:
    print("ℹ️ CLIP 模型已存在，跳過。")

# 2. 下載語言/視覺基座模型
MINIMIND_DIR = "/content/minimind-v-vla/MiniMind2-V"
if not os.path.exists(MINIMIND_DIR) or not os.path.exists(os.path.join(MINIMIND_DIR, "pytorch_model.bin")):
    print("📥 正在下載 MiniMind2-V 基座模型 (約 200MB)...")
    !git clone --depth=1 https://huggingface.co/jingyaogong/MiniMind2-V {MINIMIND_DIR}
    print("✅ MiniMind2-V 下載完成！")

# 若選用 MoE，下載官方 MiniMind2-MoE 語言權重以供初始化
MINIMIND_MOE_DIR = "/content/minimind-v-vla/MiniMind2-MoE"
if MODEL_ARCH == "MoE":
    if not os.path.exists(MINIMIND_MOE_DIR) or not os.path.exists(os.path.join(MINIMIND_MOE_DIR, "pytorch_model.bin")):
        print("📥 正在下載 MiniMind2-MoE 基座權重...")
        !git clone --depth=1 https://huggingface.co/jingyaogong/MiniMind2-MoE {MINIMIND_MOE_DIR}
        print("✅ MiniMind2-MoE 下載完成！")

# 3. (可選) 下載官方預訓練好的 Dense 權重
DOWNLOAD_OFFICIAL_PRETRAINED = True # @param {type:"boolean"}
if DOWNLOAD_OFFICIAL_PRETRAINED and MODEL_ARCH == "Dense":
    import urllib.request
    os.makedirs("VLA/models", exist_ok=True)
    sft_ckpt = "VLA/models/sft_vlm_ttt_768.pth"
    ppo_ckpt = "VLA/models/ppo_vlm_ttt.pth"
    if not os.path.exists(sft_ckpt):
        try:
            print("📥 下載官方預訓練 SFT Dense 權重...")
            urllib.request.urlretrieve("https://huggingface.co/cxf213/minimind-v-vla/resolve/main/sft_vlm_ttt_768.pth", sft_ckpt)
            print("✅ 官方 SFT 權重已就緒！")
        except Exception as e:
            print(f"⚠️ 下載失敗: {e}")


## 🎲 步驟 4：生成井字棋視覺資料集 (Dataset Generation)
透過 Minimax 演算法與自博弈混合策略，自動生成帶有：
- **棋盤渲染圖片 (RGB Image)**
- **戰術思維鏈思考 (CoT Thinking)**
- **最優動作標籤 (Optimal Action: 0~8)**

資料集格式為 Alpaca 視覺格式，包含 `train`, `val`, `test` 切分。


In [ ]:
# 設定 PYTHONPATH 以正確載入 VLA 與 envs 模組
import os
import sys

%cd /content/minimind-v-vla
os.environ["PYTHONPATH"] = f"/content/minimind-v-vla/VLA:/content/minimind-v-vla:{os.environ.get('PYTHONPATH', '')}"

NUM_SAMPLES = 5000 # @param {type:"integer"}

# 清理舊資料 (若需重新生成)
REGENERATE = False # @param {type:"boolean"}
if REGENERATE:
    os.system("rm -rf VLA/data/*")

train_data_file = "VLA/data/ttt_train_alpaca.jsonl"
if not os.path.exists(train_data_file) or REGENERATE:
    print(f"🎲 開始生成 {NUM_SAMPLES} 筆井字棋 Alpaca 視覺微調資料...")
    !python VLA/envs/generate_dataset.py         --num-samples {NUM_SAMPLES}         --out-dir VLA/data         --prefix ttt         --split-ratios 0.8,0.1,0.1         --players both
    print("✅ 資料集生成完成！")
else:
    print(f"ℹ️ 資料集已存在: {train_data_file}")


In [ ]:
# 視覺化展示生成的一筆訓練樣本
import json
from PIL import Image
import matplotlib.pyplot as plt

with open("VLA/data/ttt_train_alpaca.jsonl", "r", encoding="utf-8") as f:
    sample_line = f.readline()
    sample_data = json.loads(sample_line)

print("📝 樣本資料內容：")
print("Instruction:", sample_data.get("instruction"))
print("Input Prompt:", sample_data.get("input"))
print("Output (Thinking + Action):", sample_data.get("output"))
print("Action Index:", sample_data.get("action"))

img_path = sample_data.get("image")
if img_path and os.path.exists(img_path):
    img = Image.open(img_path)
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"Target Action: {sample_data.get('action')}")
    plt.axis('off')
    plt.show()


## 🎓 步驟 5：階段一 —— SFT 監督微調 (支援 MoE 與斷點續訓)
在此階段進行多工聯合損失訓練：
$$\mathcal{L} = \mathcal{L}_{\text{LM}} + \text{AuxLoss}_{\text{MoE}} + 0.5 \times \mathcal{L}_{\text{Action}}$$

- **若選擇 MoE**：模型會在前饋層自動構建 `MOEFeedForward`，門控層會動態學習如何將 Token 分發給最擅長的專家，同時計算負載均衡損失（AuxLoss）防止專家閒置。
- **斷點續訓**：若勾選 `RESUME_SFT = True`，會自動載入先前訓練的 Checkpoint 繼續微調。
- **背景守護 (`nohup`)**：防止分頁斷線中斷訓練。


In [ ]:
import os
import signal
import subprocess
import shutil

# 訓練超參數設定
EPOCHS = 5 # @param {type:"integer"}
BATCH_SIZE = 32 # @param {type:"integer"}
LEARNING_RATE = 1e-5 # @param {type:"number"}
NUM_WORKERS = 2 # @param {type:"integer"}
SAVE_INTERVAL = 50 # @param {type:"integer"}

# 🔄 斷點續訓控制參數
RESUME_SFT = False # @param {type:"boolean"}
AUTO_DISCOVER_CKPT = True # @param {type:"boolean"}

# 模型檔名後綴 (MoE 模式存為 _moe.pth)
CKPT_NAME = "sft_vlm_ttt_768_moe.pth" if MODEL_ARCH == "MoE" else "sft_vlm_ttt_768.pth"
CUSTOM_RESUME_PATH = f"VLA/models/{CKPT_NAME}"

RUN_IN_BACKGROUND = True # @param {type:"boolean"}
LOG_FILE = "/content/sft_train.log"
PID_FILE = "/content/sft_train.pid"

%cd /content/minimind-v-vla

# 決定續訓權重路徑
resume_arg = ""
if RESUME_SFT:
    target_ckpt = None
    if os.path.exists(CUSTOM_RESUME_PATH):
        target_ckpt = CUSTOM_RESUME_PATH
    elif AUTO_DISCOVER_CKPT:
        drive_ckpt = f"/content/drive/MyDrive/minimind_v_vla_checkpoints/{CKPT_NAME}"
        if os.path.exists(drive_ckpt):
            shutil.copy2(drive_ckpt, CUSTOM_RESUME_PATH)
            target_ckpt = CUSTOM_RESUME_PATH
            print(f"🔍 已自動從 Google Drive 還原權重以供續訓: {drive_ckpt}")
            
    if target_ckpt:
        print(f"🔄 啟用斷點續訓！載入: {target_ckpt}")
        resume_arg = f"--resume_from {target_ckpt}"
    else:
        print(f"⚠️ 未找到續訓權重 {CUSTOM_RESUME_PATH}，將從基座模型開始訓練。")

# 組合 MoE 參數
moe_flags = ""
if MODEL_ARCH == "MoE":
    moe_flags = (
        f"--use_moe "
        f"--num_experts_per_tok {NUM_EXPERTS_PER_TOK} "
        f"--n_routed_experts {N_ROUTED_EXPERTS} "
        f"--n_shared_experts {N_SHARED_EXPERTS} "
    )
    print(f"🧠 訓練命令已加入 MoE 混合專家參數！")

train_cmd = (
    f"export PYTHONPATH=/content/minimind-v-vla/VLA:/content/minimind-v-vla:$PYTHONPATH && "
    f"export SDL_VIDEODRIVER=dummy && "
    f"export WANDB_MODE=offline && "
    f"python -m VLA.SFT.train_ttt_sft "
    f"--epochs {EPOCHS} "
    f"--batch_size {BATCH_SIZE} "
    f"--learning_rate {LEARNING_RATE} "
    f"--num_workers {NUM_WORKERS} "
    f"--out_dir VLA/models "
    f"--data_path VLA/data/ttt_train_alpaca.jsonl "
    f"--val_data_path VLA/data/ttt_val_alpaca.jsonl "
    f"--save_interval {SAVE_INTERVAL} "
    f"{moe_flags} "
    f"{resume_arg}"
)

if RUN_IN_BACKGROUND:
    def is_pid_alive(p_file):
        if not os.path.exists(p_file): return False
        try:
            pid = int(open(p_file).read().strip())
            cmd = open(f"/proc/{pid}/cmdline", "rb").read().decode("utf-8", errors="ignore")
            return "train_ttt_sft" in cmd
        except Exception:
            return False

    if is_pid_alive(PID_FILE):
        print("⚠️ 訓練進程已在背景運行中！請使用下方的監控儲存格查看進度。")
    else:
        full_cmd = f"nohup bash -c '{train_cmd}' >> {LOG_FILE} 2>&1 & echo $! > {PID_FILE}"
        subprocess.run(full_cmd, shell=True, check=True)
        pid = open(PID_FILE).read().strip()
        print(f"🚀 SFT ({MODEL_ARCH}) 訓練已啟動於背景！PID: {pid}")
        print(f"📄 日誌記錄至: {LOG_FILE}")
        print("💡 您現在可安心關閉瀏覽器，訓練不會中斷。請執行下一格監控訓練日誌。")
else:
    print(f"▶️ 前台即時運行 {MODEL_ARCH} 訓練中...")
    !{train_cmd}


In [ ]:
# 📡 監控 SFT 訓練即時進度 (含背景自動同步最新 Checkpoint 到 Google Drive)
import time
import os
import shutil

PID_FILE = "/content/sft_train.pid"
LOG_FILE = "/content/sft_train.log"
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/minimind_v_vla_checkpoints"

def is_running():
    if not os.path.exists(PID_FILE): return False
    try:
        pid = int(open(PID_FILE).read().strip())
        cmd = open(f"/proc/{pid}/cmdline", "rb").read().decode("utf-8", errors="ignore")
        return "train_ttt_sft" in cmd
    except Exception:
        return False

def sync_latest_checkpoints():
    if os.path.exists("/content/drive/MyDrive"):
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        for ckpt in glob.glob("VLA/models/*.pth"):
            dst = os.path.join(DRIVE_BACKUP_DIR, os.path.basename(ckpt))
            shutil.copy2(ckpt, dst)

if not os.path.exists(LOG_FILE):
    print("⏳ 等待日誌產生...")
    time.sleep(2)

print("📡 即時監控訓練輸出 (點擊儲存格左方 ■ 即可停止畫面更新，背景進程持續運作)：")
print("-" * 60)

sync_counter = 0
try:
    with open(LOG_FILE, "r") as f:
        f.seek(0, os.SEEK_END)
        pos = max(0, f.tell() - 2000)
        f.seek(pos)
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                if not is_running():
                    print("\n🏁 訓練進程已結束！正在執行最終 Checkpoint 同步...")
                    sync_latest_checkpoints()
                    print("✅ 最終權重已備份至 Google Drive！")
                    break
                time.sleep(1)
                sync_counter += 1
                if sync_counter >= 60:
                    sync_latest_checkpoints()
                    sync_counter = 0
except KeyboardInterrupt:
    print("\n🛑 已停止即時畫面更新，背景訓練仍在安全執行中。")


In [ ]:
# 🛑 緊急手動停止 SFT 訓練 (防呆保護：避免 Run All 自動誤殺)
import os
import signal

CONFIRM_STOP = False # @param {type:"boolean"}
PID_FILE = "/content/sft_train.pid"

if not CONFIRM_STOP:
    print("ℹ️ 未勾選 CONFIRM_STOP，安全閥生效中，背景訓練不受影響。")
else:
    if os.path.exists(PID_FILE):
        try:
            pid = int(open(PID_FILE).read().strip())
            os.kill(pid, signal.SIGTERM)
            print(f"🛑 已成功向進程 {pid} 發送停止訊號。")
            os.remove(PID_FILE)
        except Exception as e:
            print(f"⚠️ 停止進程失敗: {e}")
    else:
        print("未發現運行中的 PID 記錄。")


## 🎯 步驟 6：階段二 —— PPO 線上強化學習 (支援 MoE 策略優化)
在 PPO 階段：
- 模型動態與 Minimax 演算法在線對弈
- **若為 MoE 架構**：路由門控會根據對弈局勢（開局佔角、中盤攻防、殘局堵截），自動為不同局面激活專門的專家網絡！


In [ ]:
# PPO 強化學習訓練 (支援 MoE 與斷點接續)
import os
import shutil

TOTAL_UPDATES = 200 # @param {type:"integer"}
OPPONENT_RANDOM_PROB = 0.05 # @param {type:"number"}
SEED = 42 # @param {type:"integer"}

# 🔄 PPO 續訓設定
RESUME_PPO = False # @param {type:"boolean"}
AUTO_RESTORE_FROM_DRIVE = True # @param {type:"boolean"}

%cd /content/minimind-v-vla

# 決定起點權重與輸出路徑
sft_name = "sft_vlm_ttt_768_moe.pth" if MODEL_ARCH == "MoE" else "sft_vlm_ttt_768.pth"
ppo_name = "ppo_vlm_ttt_moe.pth" if MODEL_ARCH == "MoE" else "ppo_vlm_ttt.pth"

PPO_SAVE_PATH = f"VLA/models/{ppo_name}"

if RESUME_PPO:
    START_MODEL = PPO_SAVE_PATH
    if not os.path.exists(START_MODEL) and AUTO_RESTORE_FROM_DRIVE:
        drive_ppo = f"/content/drive/MyDrive/minimind_v_vla_checkpoints/{ppo_name}"
        if os.path.exists(drive_ppo):
            shutil.copy2(drive_ppo, START_MODEL)
            print(f"🔍 自動從 Google Drive 還原前次 PPO 權重: {drive_ppo}")
    print(f"🔄 啟用 PPO 斷點續訓模式！起點: {START_MODEL}")
else:
    START_MODEL = f"VLA/models/{sft_name}"
    if not os.path.exists(START_MODEL) and AUTO_RESTORE_FROM_DRIVE:
        drive_sft = f"/content/drive/MyDrive/minimind_v_vla_checkpoints/{sft_name}"
        if os.path.exists(drive_sft):
            shutil.copy2(drive_sft, START_MODEL)
            print(f"🔍 自動從 Google Drive 還原 SFT 權重: {drive_sft}")
    print(f"▶️ 從 SFT 模型啟動 PPO 強化學習: {START_MODEL}")

if not os.path.exists(START_MODEL):
    print(f"⚠️ 警告: 未找到 {START_MODEL}，請確認前置步驟是否已生成權重。")

moe_ppo_flags = ""
if MODEL_ARCH == "MoE":
    moe_ppo_flags = (
        f"--use_moe "
        f"--num_experts_per_tok {NUM_EXPERTS_PER_TOK} "
        f"--n_routed_experts {N_ROUTED_EXPERTS} "
        f"--n_shared_experts {N_SHARED_EXPERTS} "
    )

ppo_cmd = (
    f"export PYTHONPATH=/content/minimind-v-vla/VLA:/content/minimind-v-vla:$PYTHONPATH && "
    f"export SDL_VIDEODRIVER=dummy && "
    f"python -m VLA.ppo_train "
    f"--model-path {START_MODEL} "
    f"--total-updates {TOTAL_UPDATES} "
    f"--opponent-random-prob {OPPONENT_RANDOM_PROB} "
    f"--save-path {PPO_SAVE_PATH} "
    f"--seed {SEED} "
    f"{moe_ppo_flags}"
)

print(f"🚀 開始 PPO 訓練 ({MODEL_ARCH}, Updates: {TOTAL_UPDATES})...")
!{ppo_cmd}
print(f"✅ PPO 訓練已完成，權重儲存於 {PPO_SAVE_PATH}！")


## 📊 步驟 7：模型測試與閉環基準評估 (Closed-Loop Evaluation)
評估模型在實際對弈中的真實勝率、平手率與敗率：
1. **靜態測試 (`test_ttt_model.py`)**：在 Test Set 上檢驗動作預測準確率（自動識別 MoE 架構）
2. **動態閉環對戰 (`close_loop_evaluation.py`)**：讓智能體與 Medium / Hard 對手連續進行 50 場完整對弈


In [ ]:
# 1. 靜態測試集準確率評估
%cd /content/minimind-v-vla

target_eval_name = "ppo_vlm_ttt_moe.pth" if MODEL_ARCH == "MoE" else "ppo_vlm_ttt.pth"
if not os.path.exists(f"VLA/models/{target_eval_name}"):
    target_eval_name = "sft_vlm_ttt_768_moe.pth" if MODEL_ARCH == "MoE" else "sft_vlm_ttt_768.pth"

EVAL_MODEL = f"VLA/models/{target_eval_name}"

if os.path.exists(EVAL_MODEL):
    print(f"📊 正在評估模型在測試集上的表現: {EVAL_MODEL}")
    !python -m VLA.SFT.test_ttt_model         --model_path {EVAL_MODEL}         --test_data VLA/data/ttt_test_alpaca.jsonl         --batch_size 32
else:
    print(f"⚠️ 找不到模型權重: {EVAL_MODEL}")

# 2. 動態閉環 50 場對戰評估
print("\n" + "="*50)
print(f"🎮 開始動態閉環對弈 (50 場對戰測試, 架構: {MODEL_ARCH})...")
if os.path.exists(EVAL_MODEL):
    !python VLA/close_loop_evaluation.py         --model-path {EVAL_MODEL}         --games 50         --agent-role both         --opponent medium


## 🕹️ 步驟 8：可視化單步推論與 CoT 思維鏈展示
在筆記本中直接繪製當前棋盤狀態，並讓 VLA 模型（MoE 或 Dense）即時感知圖像、輸出思維鏈推理與落子決策！


In [ ]:
import os
import sys
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

%cd /content/minimind-v-vla
os.environ["PYTHONPATH"] = f"/content/minimind-v-vla/VLA:/content/minimind-v-vla:{os.environ.get('PYTHONPATH', '')}"

from model.model_vlm import VLMConfig
from VLA.envs.model_wrapper import MiniMindVLMWithAction
from VLA.envs.tic_tac_toe_env import Difficulty, Player, TicTacToeEnv

# 選擇評估模型
model_name = "ppo_vlm_ttt_moe.pth" if MODEL_ARCH == "MoE" else "ppo_vlm_ttt.pth"
if not os.path.exists(f"VLA/models/{model_name}"):
    model_name = "sft_vlm_ttt_768_moe.pth" if MODEL_ARCH == "MoE" else "sft_vlm_ttt_768.pth"

model_path = f"VLA/models/{model_name}"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"載入模型: {model_path} (架構: {MODEL_ARCH}, 裝置: {device})")

# 初始化配置與模型
use_moe = (MODEL_ARCH == "MoE") or ("_moe" in model_path.lower())
model_config = VLMConfig(
    hidden_size=768,
    num_hidden_layers=16,
    max_seq_len=512,
    use_moe=use_moe,
    num_experts_per_tok=NUM_EXPERTS_PER_TOK if use_moe else 2,
    n_routed_experts=N_ROUTED_EXPERTS if use_moe else 4,
    n_shared_experts=N_SHARED_EXPERTS if use_moe else 1
)

vla_model = MiniMindVLMWithAction(model_config, vision_model_path="model/vision_model/clip-vit-base-patch16")
if os.path.exists(model_path):
    state_dict = torch.load(model_path, map_location=device)
    vla_model.load_state_dict(state_dict, strict=False)
vla_model.eval().to(device)

tokenizer = AutoTokenizer.from_pretrained("model")
preprocess = vla_model.processor

# 建立環境並模擬一個局面 (例如 X 佔據中宮，輪到 O)
env = TicTacToeEnv(render_mode="rgb_array", board_size=300)
env.reset()
env.board[1, 1] = Player.X.value  # X 落子於中心
board_img = env.render()
pil_img = Image.fromarray(board_img)

def format_board_text(board):
    mapping = {0: " ", 1: "X", 2: "O"}
    rows = ["[" + ", ".join([mapping[int(board[r, c])] for c in range(3)]) + "]" for r in range(3)]
    return "[" + ", ".join(rows) + "]"

desc = format_board_text(env.board)
instruction = '你是一名井字棋智能體。根據圖像與棋盤描述判斷當前局面，僅輸出 JSON {"thinking": string, "action": [row, col]}。'
player_line = "當前輪到 O 落子，請為 O 選擇最優落子位置。"
content = "\n".join([model_config.image_special_token, instruction, f"當前棋盤: {desc}。", player_line])
messages = [{"role": "user", "content": content}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
pixel_values = preprocess(pil_img).unsqueeze(0).to(device)

# 推論
with torch.no_grad():
    res = vla_model(input_ids, pixel_values=pixel_values)
    last_hidden = res.last_hidden_state[:, -1, :]
    action_logits = vla_model.action_head(last_hidden)
    predicted_action = torch.argmax(action_logits, dim=-1).item()
    row, col = divmod(predicted_action, 3)

# 繪製結果
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(pil_img)
arch_label = f"MoE ({N_ROUTED_EXPERTS} Experts, Top-{NUM_EXPERTS_PER_TOK})" if use_moe else "Dense"
ax.set_title(f"VLA [{arch_label}] 決策: 動作編號 {predicted_action} -> 座標 ({row}, {col})", fontsize=11)
ax.axis('off')
plt.show()

print(f"🎯 模型預測落子: 行 {row}, 列 {col}")


## 💾 步驟 9：權重同步至 Google Drive 與安全關機
將訓練好的模型權重（包含 MoE 權重）與訓練日誌備份至 Google Drive，確保成果永不遺失。
在完成所有訓練後，可選擇自動卸載 GPU 執行階段 (`runtime.unassign()`)，停止消耗 Colab 運算點數。


In [ ]:
# 同步權重與日誌到 Google Drive
import shutil
import glob
import os

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/minimind_v_vla_checkpoints"

if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    models = glob.glob("/content/minimind-v-vla/VLA/models/*.pth")
    for m in models:
        dst = os.path.join(DRIVE_BACKUP_DIR, os.path.basename(m))
        shutil.copy2(m, dst)
        print(f"💾 已備份權重: {os.path.basename(m)} -> {dst}")
        
    for log_f in ["/content/sft_train.log", "/content/minimind-v-vla/VLA/result"]:
        if os.path.exists(log_f):
            dst = os.path.join(DRIVE_BACKUP_DIR, os.path.basename(log_f))
            if os.path.isdir(log_f):
                shutil.copytree(log_f, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(log_f, dst)
            print(f"📄 已備份日誌/結果: {os.path.basename(log_f)}")
    print("✅ 全部重要檔案已安全同步至 Google Drive！")
else:
    print("ℹ️ Google Drive 未掛載，跳過備份。")


In [ ]:
# 🔌 自動卸載 GPU 執行階段 (省錢機制)
AUTO_SHUTDOWN = False # @param {type:"boolean"}
WAIT_FOR_TRAIN_FINISH = True # @param {type:"boolean"}

PID_FILE = "/content/sft_train.pid"

def is_train_running():
    if not os.path.exists(PID_FILE): return False
    try:
        pid = int(open(PID_FILE).read().strip())
        cmd = open(f"/proc/{pid}/cmdline", "rb").read().decode("utf-8", errors="ignore")
        return "train_ttt_sft" in cmd
    except Exception:
        return False

if AUTO_SHUTDOWN:
    if WAIT_FOR_TRAIN_FINISH and is_train_running():
        print("⏳ 正在等待訓練完成後再自動釋放 GPU...")
        while is_train_running():
            time.sleep(30)
        time.sleep(15)
        
    print("💤 正在釋放 GPU 執行階段以停止扣點...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"釋放失敗: {e}")
else:
    print("ℹ️ AUTO_SHUTDOWN 為 False，執行階段保持運行。")
